In [1]:
!pip install biopython
!pip install numpy
!pip install pandas
!pip install scipy
!pip install openpyxl
import numpy as np
from Bio.Seq import Seq
import pandas as pd
import itertools
import re
from scipy.spatial import distance
import json
from Bio.SeqUtils import MeltingTemp as mt
import random
import pprint
import string

In [57]:
#Paste gene
gene = 'ATGgcggttctctggaggctgagtgccgtttgcggtgccctaggaggccgagctctgttgcttcgaactccagtggtcagacctgctcatatctcagcatttcttcaggaccgacctatcccagaatggtgtggagtgcagcacatacacttgtcaccgagccaccattctggctccaaggctgcatctctccactggactagcgagagggttgtcagtgttttgctcctgggtctgcttccggctgcttatttgaatccttgctctgcgatggactattccctggctgcagccctcactcttcatggtcactggggccttggacaagttgttactgactatgttcatggggatgccttgcagaaagctgccaaggcagggcttttggcactttcagctttaacctttgctgggctttgctatttcaactatcacgatgtgggcatctgcaaagctgttgccatgctgtggaagctcacg'
gene = gene.upper()
gene = Seq(gene)

#add sapI sites
sapI_seq = Seq('GCTCTTC')
sapI_seqplusone = Seq('GCTCTTCC')
bsaI_seq = Seq('GGTCTC')
bsaI_seqplusone = Seq('GGTCTCT')

first_overhang = Seq('CGTC')
last_overhang = Seq('GCAT')
gene_capped = first_overhang + sapI_seqplusone + gene + sapI_seqplusone.reverse_complement() + last_overhang


In [61]:
#Get block size and initial breakpoint size
gene_size = len(gene_capped)
block_size_range = [165, 175]
max_oligo_size=250
slack = 5
block_size = block_size_range[0] + np.argmin(
    [abs(gene_size/(i+block_size_range[0])-round(gene_size/(i+block_size_range[0]))) \
         for i in range(0, block_size_range[1]-block_size_range[0])])
fragment_number = int(np.ceil(gene_size/block_size))
first_overlap = gene[0:4]
last_overlap = gene[-4:]
first_breakpoint = 0
last_breakpoint = gene_size-4

initial_breakpoints = [(first_breakpoint,block_size+slack+2,last_breakpoint)] + \
                        [(0,i-2-slack,i+block_size+slack+2,last_breakpoint) for i in range(block_size, gene_size-block_size, block_size)] + \
                        [(0,gene_size-2-slack-block_size,last_breakpoint)]


In [62]:
initial_breakpoints

[(0, 175, 500), (0, 161, 343, 500), (0, 329, 500)]

In [248]:
#Import primers
orthogonal_F_imported = pd.read_excel('./RR_orthogonalFv1_plate.xlsx')
orthogonal_R_imported = pd.read_excel('./RR_orthogonalRv1_plate.xlsx')

In [249]:
orthogonal_F = [Seq(x[-20:]) for x in orthogonal_F_imported.Sequence.values]
orthogonal_R = [Seq(x[-20:]) for x in orthogonal_R_imported.Sequence.values]

In [250]:
# remove A1
orthogonal_F = orthogonal_F[1:]
orthogonal_R = orthogonal_R[1:]

In [271]:
# get primers to estimate Q5 Tm
for (i,(x,y)) in enumerate(zip(orthogonal_F_imported.Sequence.values,orthogonal_R_imported.Sequence.values)):
    print('F'+str(i+1)+';'+x[-20:]+';R'+str(i+1)+';'+y[-20:])
                                            

F1;AGTATCTCAGCAAGGGCAAC;R1;GTTGCATCTAAGCCAAGTGC
F2;CCAGAGCTTAGGGGACATAC;R2;AAGGACTGCATACCAGGTTG
F3;GCACGCAAAAGGACATAACC;R3;ATGCTAGCTGGAACTATCGG
F4;AGACACAAGGCTGATTCCAG;R4;ACGTGAAACTGTATCGAGCC
F5;TCCAATTATACGGAGCAGGC;R5;ATTCAAGGGTTGGACGACTC
F6;TAGTTGAGAACACGAACCCG;R6;TACTGATAATTCGGACGCCC
F7;CAGACCTACGGATCTTAGCG;R7;TAAGATAGCACCACGGATGG
F8;AAGGCCCAGAAGGATACAAC;R8;AGATAGTCACGCACAAGACC
F9;TATCAATCCGGAACCAGTGC;R9;ACAACGAGCAGACCGAATAG
F10;AGTCCGACACAATGTGACAC;R10;TAACGACGTGCCGAACTTAG
F11;ACGAGATGATGCACCGATAG;R11;ATGCCATGACGACAACTAGC
F12;GACCATGCAAGGAGAGGTAC;R12;GGGTTGTCTCCTCTGATAGC
F13;TGCATAGTATCCCAACAGGG;R13;TGGGGACGACTTATAATGCC
F14;AATGCGTCATTTTACACGGC;R14;GGAAAACTAAGACAAGGCGC
F15;CGGATCGAACTTAGGTAGCC;R15;AAGGCGCTCGGATAATACTC
F16;GTTCAGAGGTACGAACCCTC;R16;CGGGAGGAAGTCTTTAGACC
F17;AAACACGTGGCAAACATTCC;R17;TCAAAGGAGCACGAACCTAC
F18;AATGCAAAGCTATTAGCGCG;R18;GCAGCGTTTTAGCCTACAAG
F19;AGCATCCGTCTAAATCTCGG;R19;CGAACGCAAAAGTCCTCAAG
F20;TAAAGAGAGGGCGTCCAATC;R20;ACCCGTATCGCATAAGGATG
F21;ACTTCGATTGGCAA

In [66]:
#Import BsaI data
bsaI_empirical = pd.read_csv('./bsaI_empirical.csv')
bsaI_empirical.index = bsaI_empirical['Overhang']
bsaI_empirical = bsaI_empirical.drop(columns=['Overhang'])
bsaI_empirical = bsaI_empirical + 1
bsaI_empirical


,AAAA,AAAC,AAAG,AAAT,AACA,AACC,AACG,AACT,AAGA,AAGC,...,TTCG,TTCT,TTGA,TTGC,TTGG,TTGT,TTTA,TTTC,TTTG,TTTT
Overhang,,,,,,,,,,,,,,,,,,,,,
TTTT,636,9,41,17,3,1,1,1,8,1,...,1,1,1,1,1,1,1,1,1,1
GTTT,4,477,5,46,1,21,1,2,1,16,...,1,1,1,1,1,1,1,1,1,1
CTTT,2,2,597,3,1,1,19,1,1,1,...,1,1,1,1,1,1,1,1,1,1
ATTT,9,5,2,643,1,1,1,7,1,2,...,1,1,1,1,1,1,1,1,1,1
TGTT,1,1,1,1,494,17,65,57,3,1,...,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACAA,1,1,1,1,1,1,1,1,1,1,...,1,1,11,3,8,480,1,1,1,1
TAAA,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,362,1,11,4
GAAA,1,1,1,1,1,1,1,1,1,1,...,1,1,1,3,1,1,6,716,2,20


In [67]:
#Import information about codon usage for mutagenesis
codons_ranked_by_usage = {
    "A": ["GCC", "GCT", "GCA", "GCG"],
    "C": ["TGC", "TGT"],
    "D": ["GAC", "GAT"],
    "E": ["GAG", "GAA"],
    "F": ["TTC", "TTT"],
    "G": ["GGC", "GGA", "GGG", "GGT"],
    "H": ["CAC", "CAT"],
    "I": ["ATC", "ATT", "ATA"],
    "K": ["AAG", "AAA"],
    "L": ["CTG", "CTC", "CTT", "TTG", "TTA", "CTA"],
    "M": ["ATG"],
    "N": ["AAC", "AAT"],
    "P": ["CCC", "CCT", "CCA", "CCG"],
    "Q": ["CAG", "CAA"],
    "R": ["CGG", "AGA", "AGG", "CGC", "CGA", "CGT"],
    "S": ["AGC", "TCC", "TCT", "AGT", "TCA", "TCG"],
    "T": ["ACC", "ACA", "ACT", "ACG"],
    "V": ["GTG", "GTC", "GTT", "GTA"],
    "W": ["TGG"],
    "Y": ["TAC", "TAT"],
}


In [133]:
#Set blacklist of inefficient codons
overhang_blacklist = []
for codon in bsaI_empirical.index.values:
    if bsaI_empirical.loc[Seq(codon),Seq(codon).reverse_complement()] < 300:
        overhang_blacklist.append(codon)
overhang_blacklist

['GTTG',
 'GGTG',
 'GTGG',
 'GGGG',
 'GCGG',
 'GGCG',
 'CCGC',
 'CGCC',
 'CCCC',
 'CACC',
 'CCAC',
 'CAAC',
 'TTAA']

In [251]:
def build_kmers(sequence, 
                ksize):
    kmers = []
    n_kmers = len(sequence) - ksize + 1

    for i in range(n_kmers):
        kmer = sequence[i:i + ksize]
        kmers.append(kmer)

    return kmers

def compute_overlaps(breakpoints, 
                     inclusion_array, 
                     gene=gene):
    
    overlaps = [[gene[val:val+4].reverse_complement(), gene[val:val+4]] for val in breakpoints]
    counter = 0
    for val in inclusion_array:
        if val == -1:
            (overlaps[counter][1],overlaps[counter+1][0]) = (overlaps[counter+1][0],overlaps[counter][1])
            counter += 1
        elif val == 0:
            overlaps[counter][1] = overlaps[counter+1][1]
            del overlaps[counter+1]
        
    return overlaps

def score_breakpoints(gene, 
                      breakpoint_pair, 
                      empirical, 
                      overhang_blacklist=overhang_blacklist):
    
    #subset empirical matrix by the set of all overlaps
    all_overlaps = []
    for breakpoint in breakpoint_pair:
        all_overlaps.append(gene[breakpoint:(breakpoint+4)])
        all_overlaps.append(gene[breakpoint:(breakpoint+4)].reverse_complement())
    all_overlaps = [str(o) for o in all_overlaps]
    if (len(np.unique(all_overlaps)) == len(all_overlaps)) & (len(set(all_overlaps).intersection(set(overhang_blacklist))) == 0):
        empirical_subset = empirical.loc[all_overlaps,all_overlaps]

        #compute fidelity score
        empirical_subset = empirical_subset/empirical_subset.sum(axis=1)
        fidelity_score = 1
        for breakpoint in breakpoint_pair:
            fidelity_score = fidelity_score * empirical_subset.loc[gene[breakpoint:(breakpoint+4)],gene[breakpoint:(breakpoint+4)].reverse_complement()]
    
    else:
        fidelity_score = 0
    
    return fidelity_score
    
def optimize_breakpoints(gene, 
                         breakpoint_pair, 
                         indices_to_shift, 
                         indices_of_array,
                         slack, 
                         empirical=bsaI_empirical, 
                         overhang_blacklist=overhang_blacklist):
    
    #compute all enrichments
    shifts = list(range(-slack,slack+1))
    if (len(indices_to_shift) > 2) | (len(indices_to_shift) < 1):
        print('Error -- too many or too few breakpoints!')
        optimum_breakpoint = breakpoint_pair
        optimum_score = 0
    elif (len(indices_to_shift) == 1): #external pair
        scores = [0]*len(shifts)
        for i,shift in enumerate(shifts):
            scores[i] = score_breakpoints(gene, breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shift] + breakpoint_pair[(indices_to_shift[0]+1):], 
                                          empirical=bsaI_empirical, overhang_blacklist=overhang_blacklist)
        
        optimum_shift = np.argmax(scores)
        optimum_breakpoint = breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shifts[optimum_shift]] + breakpoint_pair[(indices_to_shift[0]+1):]
        optimum_score = scores[optimum_shift]
        optimum_length = optimum_breakpoint[indices_of_array[1]] - optimum_breakpoint[indices_of_array[0]]
            
            
    else: #internal pair
        indices_to_shift = sorted(indices_to_shift)
        scores = np.zeros((len(shifts),len(shifts)))
        for i,shift1 in enumerate(shifts):
            for j,shift2 in enumerate(shifts):
                scores[i,j] = score_breakpoints(gene, breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shift1] + \
                                                            breakpoint_pair[(indices_to_shift[0]+1):indices_to_shift[1]] + \
                                                            [breakpoint_pair[indices_to_shift[1]]+shift2] + breakpoint_pair[(indices_to_shift[1]+1):], 
                                              empirical=bsaI_empirical, overhang_blacklist=overhang_blacklist)
                
        optimum_shift = np.unravel_index(np.argmax(scores,axis=None), scores.shape)
        optimum_breakpoint = breakpoint_pair[0:indices_to_shift[0]] + [breakpoint_pair[indices_to_shift[0]]+shifts[optimum_shift[0]]] + \
                                            breakpoint_pair[(indices_to_shift[0]+1):indices_to_shift[1]] + \
                                            [breakpoint_pair[indices_to_shift[1]]+shifts[optimum_shift[1]]] + breakpoint_pair[(indices_to_shift[1]+1):]
        optimum_score = scores[optimum_shift]
        optimum_length = optimum_breakpoint[indices_of_array[1]] - optimum_breakpoint[indices_of_array[0]] + 4
    
    return optimum_breakpoint, optimum_score, optimum_length

def optimize_gene(gene, 
                  block_size_range=block_size_range, 
                  slack=slack, 
                  empirical=bsaI_empirical, 
                  overhang_blacklist=overhang_blacklist): 
    
    #setup initial inputs to optimization
    gene_size = len(gene)
    protein_size = len(gene.translate())
        
    #exclude gene if it is too big
    if protein_size > 1000:
        print('Protein size too big!')
        
    #divide genes between 500 and 1000aa into two blocks
    elif protein_size > 550:
        print('Protein size too big! Will add two superblock (551aa+ proteins) soon.')
        
        
#         print('Protein is two superblocks.')
#         #set up superblocks, now 5-part SapI 
#         #find intermediate codon which can be thrown out of library and be used as golden-gate overlap
#         first_breakpoint = 0
#         last_breakpoint = gene_size-3
#         shifts = list(range(-slack,slack+1))
#         score_breaks = [0]*len(shifts)
#         for i,shift in enumerate(shifts):
#             codon_loc = 3*(protein_size//2+shift)
#             break_codon = gene[codon_loc:(codon_loc+3)]
#             score_breaks[i] = score_breakpoints(gene, [first_breakpoint, codon_loc, last_breakpoint], 
#                                           empirical=sapI_overlap_empirical, codon_blacklist=codon_blacklist)
        
#         optimum_shift = np.argmax(score_breaks)
#         optimum_codontobreak = 3*(protein_size//2+shifts[optimum_shift])
#         gene_first_half = gene[0:optimum_codontobreak+3]
#         gene_second_half = gene[optimum_codontobreak:]
        
#         optimum_breakpoints = []
#         optimum_scores = []
#         optimum_lengths = []
#         oligo_array_indices = []
        
#         for i, gene_block in enumerate([gene_first_half,gene_second_half]):
#             #now, set up blocks for each half and perform two optimizations
#             gene_block_size = len(gene_block)
#             block_size = block_size_range[0] + np.argmin(
#                 [abs(gene_block_size/(i+block_size_range[0])-round(gene_block_size/(i+block_size_range[0]))) \
#                      for i in range(0, block_size_range[1]-block_size_range[0])])
#             fragment_number = int(np.ceil(gene_block_size/block_size))
#             if i == 0:
#                 pre_first_breakpoint = []
#                 first_breakpoint = [0]
#                 last_breakpoint = [optimum_codontobreak]
#                 post_last_breakpoint = [gene_size-3]
#             elif i == 1:
#                 pre_first_breakpoint = [0]
#                 first_breakpoint = [optimum_codontobreak]
#                 last_breakpoint = [gene_size-3]
#                 post_last_breakpoint = []
#             else:
#                 print('Too many superblocks!')
#             initial_breakpoints = [pre_first_breakpoint + first_breakpoint + \
#                                    [first_breakpoint[0]+block_size+slack+2] + last_breakpoint + post_last_breakpoint] + \
#                                 [pre_first_breakpoint + first_breakpoint + \
#                                  [first_breakpoint[0]+j-2-slack, first_breakpoint[0]+j+block_size+slack+2] + last_breakpoint + post_last_breakpoint \
#                                      for j in range(block_size, gene_block_size-block_size-slack-3, block_size)] + \
#                                 [pre_first_breakpoint + first_breakpoint + \
#                                  [first_breakpoint[0]+gene_block_size-2-slack-block_size] + last_breakpoint + post_last_breakpoint]
            
#             #optimize each breakpoint
#             for k,breakpoint in enumerate(initial_breakpoints):
#                 if len(breakpoint) == 4:
#                     indices_of_array = [0+i, 1+i] if k==0 else [1+i, 2+i]
#                     optimum_breakpoint, optimum_score, optimum_length = optimize_breakpoints(gene, breakpoint, [1+i], indices_of_array,
#                                                                 slack, empirical=sapI_overlap_empirical, codon_blacklist=codon_blacklist)
#                     optimum_breakpoints.append(optimum_breakpoint)
#                     optimum_scores.append(optimum_score)
#                     optimum_lengths.append(optimum_length)
#                     oligo_array_indices.append(indices_of_array)
                    
#                 else:
#                     indices_of_array = [1+i, 2+i]
#                     optimum_breakpoint, optimum_score, optimum_length = optimize_breakpoints(gene, breakpoint, [1+i, 2+i], indices_of_array, 
#                                                                     slack, empirical=sapI_overlap_empirical, codon_blacklist=codon_blacklist)
#                     optimum_breakpoints.append(optimum_breakpoint)
#                     optimum_scores.append(optimum_score)
#                     optimum_lengths.append(optimum_length)
#                     oligo_array_indices.append(indices_of_array)
        
    else:
        #gene is one superblock 
        #print('Protein is one superblock.')
        block_size = block_size_range[0] + np.argmin(
            [abs(gene_size/(i+block_size_range[0])-round(gene_size/(i+block_size_range[0]))) \
                 for i in range(0, block_size_range[1]-block_size_range[0])])
        fragment_number = int(np.ceil(gene_size/block_size))
        first_breakpoint = 0
        last_breakpoint = gene_size-4
        initial_breakpoints = [[first_breakpoint,block_size+slack+2,last_breakpoint]] + \
                                [[0,i-2-slack,i+block_size+slack+2,last_breakpoint] for i in range(block_size, gene_size-block_size-slack-4, block_size)] + \
                                [[0,gene_size-2-slack-block_size,last_breakpoint]]
    
        #optimize each breakpoint
        optimum_breakpoints = []
        optimum_scores = []
        optimum_lengths = []
        oligo_array_indices = []
        for k,breakpoint in enumerate(initial_breakpoints):
            if len(breakpoint) == 3:
                indices_of_array = [0, 1] if k==0 else [1, 2]
                optimum_breakpoint, optimum_score, optimum_length = optimize_breakpoints(gene, breakpoint, [1], indices_of_array,
                                                            slack, empirical=bsaI_empirical, overhang_blacklist=overhang_blacklist)
                optimum_breakpoints.append(optimum_breakpoint)
                optimum_scores.append(optimum_score)
                optimum_lengths.append(optimum_length)
                oligo_array_indices.append(indices_of_array)
            else:
                indices_of_array = [1, 2]
                optimum_breakpoint, optimum_score, optimum_length = optimize_breakpoints(gene, breakpoint, [1, 2], indices_of_array,
                                                            slack, empirical=bsaI_empirical, overhang_blacklist=overhang_blacklist)
                optimum_breakpoints.append(optimum_breakpoint)
                optimum_scores.append(optimum_score)
                optimum_lengths.append(optimum_length)
                oligo_array_indices.append(indices_of_array)
    
    optimum_overlaps = [[str(gene[t:(t+4)]) for t in s] for s in optimum_breakpoints]
    if all([s >= 0.95 for s in optimum_scores]):
        print('All regions are high fidelity!')
    elif all([s >= 0.9 for s in optimum_scores]):
        print('Some regions are medium fidelity.')
    else:
        print('Some regions are low fidelity. Look closer')
        
    return optimum_breakpoints, optimum_overlaps, optimum_scores, optimum_lengths, oligo_array_indices

def generate_primer(DNA_seq,
                     Fwd=True,
                     extendtoCG=False,
                     smallest_primer_size=16,
                     largest_primer_size=30,
                     Tm=55):
    
    #Setup melting temperature arrays
    melt_temp_array = np.zeros(largest_primer_size-smallest_primer_size+1)
    
    if Fwd:
        DNA_seq_touse = DNA_seq
    else:
        DNA_seq_touse = DNA_seq.reverse_complement()
            
    #Make melting temperature arrays
    primer_length = 0
    for i in range(smallest_primer_size,largest_primer_size+1):
        melt_temp_array[i-smallest_primer_size] = mt.Tm_NN(DNA_seq_touse[0:i])
        
        #Pick F primer when Tm is first >F_Tm
        if (melt_temp_array[i-smallest_primer_size] >= Tm) & (primer_length==0):
            primer_length = i
    
    #If Tm isnt high enough after max bases, just set primer length to be max and hope it works
    if (primer_length == 0):
        primer_length = largest_primer_size
        
    if extendtoCG:
        while ((DNA_seq_touse[primer_length-1] == 'A') | (DNA_seq_touse[primer_length-1] == 'T')) & \
                    (primer_length < largest_primer_size):
            primer_length += 1
    
    return DNA_seq_touse[0:primer_length]

def make_all_mutations(region_name,
                       region,
                       region_flanks=[Seq(''),Seq('')],
                       nt_start=0, #zero-indexed!
                       wt_only=False,
                       synonymous=True,
                       stops='TAA',
                       all3ntdeletions=True,
                       codons_ranked_by_usage=codons_ranked_by_usage):
    
    oligo_array = {}
    #Check that region has size divisible by three
    if (len(region)/3 != len(region)//3) | (nt_start/3 != nt_start//3):
        print('Region is not translatable!')
        
    else:
        #add wt seq to oligo array
        oligo_name = region_name + '_WT'
        wt_seq = \
            region_flanks[0] + region + region_flanks[1]
        oligo_array[oligo_name] = wt_seq
        
        if not wt_only:
                    
            #loop over amino acids
            for j in range(0,len(region),3):

                #add all missense variants
                aa = region[j:(j+3)].translate()
                for aa_to in codons_ranked_by_usage.keys():
                    if aa_to != aa:
                        oligo_name = region_name + '_' + str(aa) + str((nt_start+j)//3+1) + str(aa_to)
                        seq_to_append = \
                            region_flanks[0] + \
                            region[0:j] + Seq(codons_ranked_by_usage[aa_to][0]) + \
                            region[(j+3):] + \
                            region_flanks[1]
                        oligo_array[oligo_name] = seq_to_append

                #add synonymous variant if True and if possible, 
                # using the most common codon that is NOT the codon in the gene
                if synonymous:
                    if len(codons_ranked_by_usage[aa]) > 1:
                        oligo_name = region_name + '_' + str(aa) + str((nt_start+j)//3+1) + str(aa)
                        possible_codons = codons_ranked_by_usage[aa].copy()
                        possible_codons.remove(region[j:(j+3)])
                        seq_to_append = \
                            region_flanks[0] + \
                            region[0:j] + Seq(possible_codons[0]) + \
                            region[(j+3):] + \
                            region_flanks[1]
                        oligo_array[oligo_name] = seq_to_append

                #add stops if true
                if stops:
                    oligo_name = region_name + '_' + str(aa) + str((nt_start+j)//3+1) + 'X'
                    seq_to_append = \
                        region_flanks[0] + \
                        region[0:j] + Seq(stops) + \
                        region[(j+3):] + \
                        region_flanks[1]
                    oligo_array[oligo_name] = seq_to_append

                #add all 3nt deletions if True
                if all3ntdeletions:
                    for k in range(0,3):
                        if j+k+3 <= len(region):
                            oligo_name = region_name + '_' + 'del' + str(nt_start+j+k+1)
                            seq_to_append = \
                                region_flanks[0] + \
                                region[0:(j+k)] + \
                                region[(j+k+3):] + \
                                region_flanks[1]
                            oligo_array[oligo_name] = seq_to_append
        
    return oligo_array


def write_oligo_library(genes,
                        oligo_file='./oligo_test.csv',
                        primer_file='./primer_test.tsv',
                        gbl_file='./gbl_test.tsv',
                        primer_set_F=orthogonal_F,
                        primer_set_R=orthogonal_R,
                        codons_ranked_by_usage=codons_ranked_by_usage,
                        block_size_range=block_size_range, 
                        max_oligo_size=max_oligo_size,
                        slack=slack, 
                        empirical=bsaI_empirical, 
                        overhang_blacklist=overhang_blacklist,
                        wt_only=False,
                        synonymous=True,
                        stops='TAA',
                        all3ntdeletions=True,
                        smallest_primer_size=16,
                        largest_primer_size=30,
                        Tm=55,
                        extendtoCG=True,
                        bsaI_firstoverlap='CGTC',
                        bsaI_lastoverlap='GCAT',
                        all_blocks=True,
                        blocks_to_include=False,
                        sapIcapF=True,
                        sapIcapR=True):
    
    #Split up primer set into F and R primers, cannot do more than 82 sublibraries
    oligo_primer_counter = 0
    oligo_array = {}
    amp_primers = {}
    gblocks = {}
    
    #Convert genes to Seq and genes to list
    gene_names = list(genes.keys())
    genes = [Seq(genes[gene_name]) for gene_name in gene_names]
    
    #SapI and BsaI site sequences
    sapI_seq = Seq('GCTCTTC')
    sapI_seqplusone = Seq('GCTCTTCC')
    bsaI_seq = Seq('GGTCTC')
    bsaI_seqplusone = Seq('GGTCTCT')
    pcr_capseq = Seq('GGCTAC') + bsaI_seqplusone
    gbl_capseq_F = Seq('CCGCGTGATTACGAGTCG') + pcr_capseq
    gbl_capseq_R = Seq('GGGTTAGCAAGTGGCAGCCT') + pcr_capseq
    
    for r,gene in enumerate(genes):
        
        print('Processing gene ' + str(r+1))
        gene_name = gene_names[r]
        
        #exclude if gene size is not divisible by three
        if len(gene)/3 != len(gene)//3:
            print('Gene length is not divisible by 3!')
    
        #exclude if there is a SapI site in the gene
        elif any([True for kmer in build_kmers(gene, len(sapI_seq)) if kmer==sapI_seq]) | \
            any([True for kmer in build_kmers(gene.reverse_complement(), len(sapI_seq)) if kmer==sapI_seq]):
            print('Gene has SapI site!')
        
        #exclude if there is a BsaI site in the gene
        elif any([True for kmer in build_kmers(gene, len(bsaI_seq)) if kmer==bsaI_seq]) | \
            any([True for kmer in build_kmers(gene.reverse_complement(), len(bsaI_seq)) if kmer==bsaI_seq]):
            print('Gene has BsaI site!')
            
        else:
            print('Gene has no SapI or BsaI sites! Performing GoldenGate optimization...')
            
            #cap gene with BsaI breakpoints and possible SapI sites 
            if sapIcapF & sapIcapR:
                gene_capped = bsaI_firstoverlap + sapI_seqplusone + \
                        gene + sapI_seqplusone.reverse_complement() + bsaI_lastoverlap
                capping_length_F = len(bsaI_firstoverlap + sapI_seqplusone)
                capping_length_R = len(sapI_seqplusone.reverse_complement() + bsaI_lastoverlap)
            elif sapIcapF:
                gene_capped = bsaI_firstoverlap + sapI_seqplusone + \
                        gene + bsaI_lastoverlap
                capping_length_F = len(bsaI_firstoverlap + sapI_seqplusone)
                capping_length_R = len(bsaI_lastoverlap)
            elif sapIcapR:
                gene_capped = bsaI_firstoverlap + \
                        gene + sapI_seqplusone.reverse_complement() + bsaI_lastoverlap
                capping_length_F = len(bsaI_firstoverlap)
                capping_length_R = len(sapI_seqplusone.reverse_complement() + bsaI_lastoverlap)
            else:
                gene_capped = bsaI_firstoverlap + gene + bsaI_lastoverlap
                capping_length_F = len(bsaI_firstoverlap)
                capping_length_R = len(bsaI_lastoverlap)
            
            #Optimize gene
            optimum_breakpoints, optimum_overlaps, optimum_scores, optimum_lengths, oligo_array_indices = \
            optimize_gene(gene_capped, 
                      block_size_range=block_size_range, 
                      slack=slack, 
                      empirical=bsaI_empirical, 
                      overhang_blacklist=overhang_blacklist)
            pprint.pprint({'Optimum Breakpoints': optimum_breakpoints, 
                   'Optimum Overlaps': optimum_overlaps, 
                   'Optimum Scores': optimum_scores})
            
            #add primers for gene_F and gene_R that are repeated constantly throughout the PCRs
            #note: should probably prevalidate these primers!
            F_primer = generate_primer(gene,
                                       Fwd=True,
                                       extendtoCG=extendtoCG,
                                       smallest_primer_size=smallest_primer_size,
                                       largest_primer_size=largest_primer_size,
                                       Tm=Tm)
            F_primer = pcr_capseq + bsaI_firstoverlap + sapI_seqplusone + F_primer
            amp_primers[gene_name+'_gene'+'_ampF'] = F_primer
            R_primer = generate_primer(gene,
                                       Fwd=False,
                                       extendtoCG=extendtoCG,
                                       smallest_primer_size=smallest_primer_size,
                                       largest_primer_size=largest_primer_size,
                                       Tm=Tm)
            R_primer = pcr_capseq + Seq(bsaI_lastoverlap).reverse_complement() + \
                        sapI_seqplusone + R_primer
            amp_primers[gene_name+'_gene'+'_ampR'] = R_primer
            
            #make oligos, primers, gblocks for each block
            for i,breakpoint in enumerate(optimum_breakpoints):
                
                #find indices of breakpoint that correspond to oligo vs need to be PCRed/gblock
                pcr_indices = [[j,j+1] for j in range(len(breakpoint)-1)]
                pcr_indices.remove(oligo_array_indices[i])
                
                #find mutagenic window of oligo
                oligo_breaks = [breakpoint[j] for j in oligo_array_indices[i]]
                oligo_mutagenic_window = [int(3*np.ceil(max(oligo_breaks[0]+4-capping_length_F,3)/3)), int(3*np.floor(min(oligo_breaks[1]-capping_length_F,len(gene)-1)/3))]
                
                #add primers and gblocks
                for k,pcr_index in enumerate(pcr_indices):
                    piece_name = gene_name + '_block' + str(i+1) + '_s' + str(k+1)
                    pcr_breaks = [breakpoint[j] for j in pcr_index]
                    
                    if (all_blocks == True) | ((i+1) in blocks_to_include[r] if blocks_to_include != False else True): #subset on allowed blocks
                        
                        #get pcr primers
                        print(pcr_breaks)
                        if pcr_breaks[0] == breakpoint[0]: #Fragment beginning at gene start 
                            R_primer = generate_primer(gene_capped[pcr_breaks[0]:(pcr_breaks[1]+4)],
                                                       Fwd=False,
                                                       extendtoCG=extendtoCG,
                                                       smallest_primer_size=smallest_primer_size,
                                                       largest_primer_size=largest_primer_size,
                                                       Tm=Tm)
                            R_primer = pcr_capseq + R_primer
                            amp_primers[piece_name+'_ampR'] = R_primer
                        elif pcr_breaks[1] == breakpoint[-1]: #Fragment ending at gene end
                            F_primer = generate_primer(gene_capped[pcr_breaks[0]:(pcr_breaks[1]+4)],
                                                       Fwd=True,
                                                       extendtoCG=extendtoCG,
                                                       smallest_primer_size=smallest_primer_size,
                                                       largest_primer_size=largest_primer_size,
                                                       Tm=Tm)
                            F_primer = pcr_capseq + F_primer
                            amp_primers[piece_name+'_ampF'] = F_primer
                        else:
                            F_primer = generate_primer(gene_capped[pcr_breaks[0]:(pcr_breaks[1]+4)],
                                                       Fwd=True,
                                                       extendtoCG=extendtoCG,
                                                       smallest_primer_size=smallest_primer_size,
                                                       largest_primer_size=largest_primer_size,
                                                       Tm=Tm)
                            F_primer = pcr_capseq + F_primer
                            R_primer = generate_primer(gene_capped[pcr_breaks[0]:(pcr_breaks[1]+4)],
                                                       Fwd=False,
                                                       extendtoCG=extendtoCG,
                                                       smallest_primer_size=smallest_primer_size,
                                                       largest_primer_size=largest_primer_size,
                                                       Tm=Tm)
                            R_primer = pcr_capseq + R_primer
                            amp_primers[piece_name+'_ampF'] = F_primer
                            amp_primers[piece_name+'_ampR'] = R_primer

                        #make gblocks
                        gbl = gbl_capseq_F + gene_capped[pcr_breaks[0]:(pcr_breaks[1]+4)] + gbl_capseq_R.reverse_complement()
                        gblocks[piece_name] = gbl

                        #add oligos to oligo array
                        add_on_array = make_all_mutations(gene_name + '_block' + str(i+1),
                                           gene[oligo_mutagenic_window[0]:oligo_mutagenic_window[1]],
                                           region_flanks=[primer_set_F[oligo_primer_counter] + \
                                                          bsaI_seqplusone + \
                                                          gene_capped[oligo_breaks[0]:(oligo_mutagenic_window[0]+capping_length_F)] ,
                                                          gene_capped[(oligo_mutagenic_window[1]+capping_length_F):(oligo_breaks[1]+4)] + \
                                                          bsaI_seqplusone.reverse_complement() + \
                                                          primer_set_R[oligo_primer_counter].reverse_complement()],
                                           nt_start=oligo_mutagenic_window[0],
                                           wt_only=wt_only,
                                           synonymous=synonymous,
                                           stops=stops,
                                           all3ntdeletions=all3ntdeletions,
                                           codons_ranked_by_usage=codons_ranked_by_usage)
                        oligo_array.update(add_on_array)
                        oligo_primer_counter += 1
                    
    #Check that max oligo is less than the max oligo length
    if sum([len(s)>max_oligo_size for s in oligo_array.values()]) == 0:
        print('All oligos are below the maximum 250bp!')
    else:
        print('Some oligos are TOO BIG!')
                
    #Remove any oligos with additional BsaI sites
    bad_oligos = []
    for name,oligo in oligo_array.items():
#         sapI_F = sum([True for kmer in build_kmers(oligo, len(sapI_seq)) if kmer==sapI_seq])
#         sapI_R = sum([True for kmer in build_kmers(oligo.reverse_complement(), len(sapI_seq)) if kmer==sapI_seq])
#         if (sapI_F != 1) | (sapI_R != 1):
#             bad_oligos.append(name)
        bsaI_F = sum([True for kmer in build_kmers(oligo, len(bsaI_seq)) if kmer==bsaI_seq])
        bsaI_R = sum([True for kmer in build_kmers(oligo.reverse_complement(), len(bsaI_seq)) if kmer==bsaI_seq])
        if (bsaI_F != 1) | (bsaI_R != 1):
            bad_oligos.append(name)
    for oligo_name in bad_oligos:
        del oligo_array[oligo_name]
    print(str(len(bad_oligos)) + ' oligos deleted due to errant restriction sites.')
    
    #Remove any duplicate oligos
    new_dict = {}
    seen_values = set()
    counter=0
    for key, value in oligo_array.items():
        if value not in seen_values:
            new_dict[key] = value
            seen_values.add(value)
        else:
            counter += 1
    print(str(counter) + ' oligos removed due to duplication.')
    oligo_array = new_dict
    del new_dict
    
    #write oligo array to file
    with open(oligo_file, 'w') as f:
        for key in oligo_array.keys():
            f.write("%s,%s\n"%(key,oligo_array[key]))
    f.close()
            
    #write primers to file
    primer_order_sheet = []
    for key in amp_primers.keys():
        primer_order_sheet.append(key + '\t' + \
                 str(amp_primers[key]) + \
                 '\t' + '25nm' + '\t' + 'STD\n')
    print(*primer_order_sheet)
    with open(primer_file, 'w') as f:
        for line in primer_order_sheet:
            f.write(line)
    f.close()
    
    #write gblocks to file
    gblock_order_sheet = []
    for key in gblocks.keys():
        gblock_order_sheet.append(key + '\t' + \
                 str(gblocks[key]) + '\n')
    print(*gblock_order_sheet)
    with open(gbl_file, 'w') as f:
        for line in gblock_order_sheet:
            f.write(line)
    f.close()
    
    return oligo_array,amp_primers,gblocks
                

In [147]:
optimize_gene(gene_capped)


Some regions are medium fidelity.


([[0, 174, 1463],
  [0, 161, 342, 1463],
  [0, 321, 505, 1463],
  [0, 491, 670, 1463],
  [0, 657, 829, 1463],
  [0, 813, 1000, 1463],
  [0, 985, 1164, 1463],
  [0, 1150, 1330, 1463],
  [0, 1296, 1463]],
 [['CGTC', 'TTCT', 'GCAT'],
  ['CGTC', 'TCCC', 'AAGA', 'GCAT'],
  ['CGTC', 'CAGA', 'TCCT', 'GCAT'],
  ['CGTC', 'TTTC', 'ACGA', 'GCAT'],
  ['CGTC', 'TCTT', 'GGGA', 'GCAT'],
  ['CGTC', 'AAGA', 'TGGA', 'GCAT'],
  ['CGTC', 'ACTA', 'AAGA', 'GCAT'],
  ['CGTC', 'TTTC', 'AGGA', 'GCAT'],
  ['CGTC', 'GTCA', 'GCAT']],
 [0.9693578138231524,
  0.9473011537441497,
  0.9493176714669691,
  0.9494231834409504,
  0.9473011537441497,
  0.9490519875816659,
  0.9488266764221456,
  0.9502978746717136,
  0.9672689252729983],
 [174, 185, 188, 183, 176, 191, 183, 184, 167],
 [[0, 1], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2]])

In [119]:
oligo_array,amp_primers,gblocks = write_oligo_library({'SDHD':gene},
                                                      oligo_file='./bsaI_test/sdhd_oligos.csv',
                                                      primer_file='./bsaI_test/sdhd_primers.tsv',
                                                      gbl_file='./bsaI_test/sdhd_gblocks.tsv')


Processing gene 1
Gene has no SapI or BsaI sites! Performing GoldenGate optimization...
Some regions are medium fidelity.
{'Optimum Breakpoints': [[0, 179, 500], [0, 160, 346, 500], [0, 332, 500]],
 'Optimum Overlaps': [['CGTC', 'TTCT', 'GCAT'],
                      ['CGTC', 'ACTT', 'CTGA', 'GCAT'],
                      ['CGTC', 'TGGA', 'GCAT']],
 'Optimum Scores': [0.9693578138231524, 0.9487813139974746, 0.9683339822030734]}
([0, 179], [3, 165])
([160, 346], [153, 333])
([332, 500], [324, 477])
All oligos are below the maximum 250bp!
18 oligos deleted due to errant restriction sites.
124 oligos removed due to duplication.
SDHD_gene_ampF	GGCTACGGTCTCTCGTCGCTCTTCCATGGCGGTTCTCTGGAGGC	25nm	STD
 SDHD_gene_ampR	GGCTACGGTCTCTATGCGCTCTTCCCGTGAGCTTCCACAGCATGG	25nm	STD
 SDHD_block1_s1_ampF	GGCTACGGTCTCTTTCTGGCTCCAAGGCTGCATC	25nm	STD
 SDHD_block2_s1_ampR	GGCTACGGTCTCTAAGTGTATGTGCTGCACTCCAC	25nm	STD
 SDHD_block2_s2_ampF	GGCTACGGTCTCTCTGACTATGTTCATGGGGATGCC	25nm	STD
 SDHD_block3_s1_ampR	GGCTACGG

In [ ]:
# DO EVAN'S GENES

In [199]:
# HIST1H1E - ntag
hist1h1e = Seq('''ATGTCCGAGACTGCGCCTGCCGCGCCCGCTGCTCCGGCCCCTGCCGAGAAGACTCCCGTGAAGAAGAAGGCCCG
CAAGTCTGCAGGTGCGGCCAAGCGCAAAGCGTCTGGGCCCCCGGTGTCCGAGCTCATTACTAAAGCTGTTGCCGCCTCCAAGGAGCGC
AGCGGCGTATCTTTGGCCGCTCTCAAGAAAGCGCTGGCAGCCGCTGGCTATGACGTGGAGAAGAACAACAGCCGCATCAAGCTGGGAC
TCAAGAGCCTGGTGAGCAAGGGCACCCTGGTGCAGACCAAGGGCACCGGCGCGTCGGGTTCCTTCAAACTCAACAAGAAGGCGGCCTC
TGGGGAAGCCAAGCCTAAGGCTAAAAAGGCAGGCGCGGCCAAGGCCAAGAAGCCAGCAGGAGCGGCGAAGAAGCCCAAGAAGGCGACG
GGGGCGGCCACCCCCAAGAAGTCCGCCAAGAAGACCCCAAAGAAGGCGAAGAAGCCGGCTGCAGCTGCTGGAGCCAAAAAAGCGAAAA
GCCCGAAAAAGGCGAAAGCAGCCAAGCCAAAAAAGGCGCCCAAGAGCCCAGCGAAGGCCAAAGCAGTTAAACCCAAGGCGGCTAAACC
AAAGACCGCCAAGCCCAAGGCAGCCAAGCCAAAGAAGGCGGCAGCCAAGAAAAAGTAG'''.replace('\n',''))
gene_capped = first_overhang + sapI_seqplusone + hist1h1e + sapI_seqplusone.reverse_complement() + last_overhang
optimize_gene(gene_capped,
             block_size_range = [165,175])



All regions are high fidelity!


([[0, 183, 680], [0, 166, 353, 680], [0, 336, 521, 680], [0, 506, 680]],
 [['CGTC', 'TCTT', 'GCAT'],
  ['CGTC', 'AGGA', 'GGAA', 'GCAT'],
  ['CGTC', 'AAGA', 'GAAA', 'GCAT'],
  ['CGTC', 'TGGA', 'GCAT']],
 [0.9695086923749419,
  0.9503362656703902,
  0.9500476572285086,
  0.9683339822030734],
 [183, 191, 189, 174],
 [[0, 1], [1, 2], [1, 2], [1, 2]])

In [198]:
# RPS19 - ctag
rps19 = Seq('''ATGCCTGGAGTTACTGTAAAAGACGTGAACCAGCAGGAGTTCGTCAGAGCTCTGGCAGCCTTCCTCAAAAAGTCC
GGGAAGCTGAAAGTCCCCGAATGGGTGGATACCGTCAAGCTGGCCAAGCACAAAGAGCTTGCTCCCTACGATGAGAACTGGTTCTACAC
GCGAGCTGCTTCCACAGCGCGGCACCTGTACCTCCGGGGTGGCGCTGGGGTTGGCTCCATGACCAAGATCTATGGGGGACGTCAGAGAA
ACGGCGTCATGCCCAGCCACTTCAGCCGAGGCTCCAAGAGTGTGGCCCGCCGGGTCCTCCAAGCCCTGGAGGGGCTGAAAATGGTGGAA
AAGGACCAAGATGGCGGCCGCAAACTGACACCTCAGGGACAAAGAGATCTGGACAGAATCGCCGGACAGGTGGCAGCTGCCAACAAGAAGCATACG'''.replace('\n',''))
gene_capped = first_overhang + sapI_seqplusone + rps19 + sapI_seqplusone.reverse_complement() + last_overhang
optimize_gene(gene_capped,
             block_size_range = [165,175])



Some regions are medium fidelity.


([[0, 168, 458], [0, 160, 341, 458], [0, 286, 458]],
 [['CGTC', 'TTCT', 'GCAT'],
  ['CGTC', 'AGAA', 'GAAA', 'GCAT'],
  ['CGTC', 'TCAG', 'GCAT']],
 [0.9693578138231524, 0.949841214886651, 0.9690333618956276],
 [168, 185, 172],
 [[0, 1], [1, 2], [1, 2]])

In [253]:
# SPOP - ntag
spop = Seq('''ATGTCAAGGGTTCCAAGTCCTCCACCTCCGGCAGAAATGTCGAGTGGCCCCGTAGCTGAGAGTTGGTGCTACACACAGATCAAGGTAGTGAAATTCTCCTACATGTGGACCATCAATAACTTTAGCTTTTGCCGGGAGGAAATGGGTGAAGTCATTAAAAGTTCTACATTTTCATCAGGAGCAAATGATAAACTGAAATGGTGTTTGCGAGTAAACCCCAAAGGGTTAGATGAAGAAAGCAAAGATTACCTGTCACTTTACCTGTTACTGGTCAGCTGTCCAAAGAGTGAAGTTCGGGCAAAATTCAAATTCTCCATCCTGAATGCCAAGGGAGAAGAAACCAAAGCTATGGAGAGTCAACGGGCATATAGGTTTGTGCAAGGCAAAGACTGGGGATTCAAGAAATTCATCCGTAGAGATTTTCTTTTGGATGAGGCCAACGGGCTTCTCCCTGATGACAAGCTTACCCTCTTCTGCGAGGTGAGTGTTGTGCAAGATTCTGTCAACATTTCTGGCCAGAATACCATGAACATGGTAAAGGTTCCTGAGTGCCGGCTGGCAGATGAGTTAGGAGGACTGTGGGAGAATTCCCGGTTCACAGACTGCTGCTTGTGTGTTGCCGGCCAGGAATTCCAGGCTCACAAGGCTATCTTAGCAGCTCGTTCTCCGGTTTTTAGTGCCATGTTTGAACATGAAATGGAGGAGAGCAAAAAGAATCGAGTTGAAATCAATGATGTGGAGCCTGAAGTTTTTAAGGAAATGATGTGCTTCATTTACACGGGGAAGGCTCCAAACCTCGACAAAATGGCTGATGATTTGCTGGCAGCTGCTGACAAGTATGCCCTGGAGCGCTTAAAGGTCATGTGTGAGGATGCCCTCTGCAGTAACCTGTCCGTGGAGAACGCTGCAGAAATTCTCATCCTGGCCGACCTCCACAGTGCAGATCAGTTGAAAACTCAGGCAGTGGATTTCATCAACTATCATGCTTCGGATGTCTTGGAAACATCTGGGTGGAAGTCAATGGTGGTGTCACATCCCCACTTGGTGGCTGAGGCATACCGCTCTCTGGCTTCAGCACAGTGCCCTTTTCTGGGACCCCCACGCAAACGCCTGAAGCAATCCTAG'''.replace('\n',''))
gene_capped = first_overhang + sapI_seqplusone + spop + sapI_seqplusone.reverse_complement() + last_overhang
optimize_gene(gene_capped,
             block_size_range = [165,175])


Some regions are medium fidelity.


([[0, 173, 1145],
  [0, 160, 333, 1145],
  [0, 321, 505, 1145],
  [0, 483, 671, 1145],
  [0, 654, 831, 1145],
  [0, 820, 998, 1145],
  [0, 980, 1145]],
 [['CGTC', 'TTCT', 'GCAT'],
  ['CGTC', 'AAGT', 'AATG', 'GCAT'],
  ['CGTC', 'TTCT', 'AAGA', 'GCAT'],
  ['CGTC', 'TTCT', 'TCGT', 'GCAT'],
  ['CGTC', 'AAGG', 'CTGG', 'GCAT'],
  ['CGTC', 'CTGA', 'TTCG', 'GCAT'],
  ['CGTC', 'TTTC', 'GCAT']],
 [0.9693578138231524,
  0.9475010775840722,
  0.9504530934571519,
  0.9498283531731762,
  0.9452174324401622,
  0.9485573803400719,
  0.9690616370636065],
 [173, 177, 188, 192, 181, 182, 165],
 [[0, 1], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2]])

In [196]:
# AKT - ntag
akt = Seq('''ATGAGCGACGTGGCTATTGTGAAGGAGGGTTGGCTGCACAAACGAGGGGAGTACATCAAGA
CCTGGCGGCCACGCTACTTCCTCCTCAAGAATGATGGCACCTTCATTGGCTACAAGGAGCGGCCGCAGGATGTGG
ACCAACGTGAGGCTCCCCTCAACAACTTCTCTGTGGCGCAGTGCCAGCTGATGAAGACGGAGCGGCCCCGGCCCA
ACACCTTCATCATCCGCTGCCTGCAGTGGACCACTGTCATCGAACGCACCTTCCATGTGGAGACTCCTGAGGAGC
GGGAGGAGTGGACAACCGCCATCCAGACTGTGGCTGACGGCCTCAAGAAGCAGGAGGAGGAGGAGATGGACTTCC
GGTCGGGCTCACCCAGTGACAACTCAGGGGCTGAAGAGATGGAGGTGTCCCTGGCCAAGCCCAAGCACCGCGTGA
CCATGAACGAGTTTGAGTACCTGAAGCTGCTGGGCAAGGGCACTTTCGGCAAGGTGATCCTGGTGAAGGAGAAGG
CCACAGGCCGCTACTACGCCATGAAGATCCTCAAGAAGGAAGTCATCGTGGCCAAGGACGAGGTGGCCCACACAC
TCACCGAGAACCGCGTCCTGCAGAACTCCAGGCACCCCTTCCTCACAGCCCTGAAGTACTCTTTCCAGACCCACG
ACCGCCTCTGCTTTGTCATGGAGTACGCCAACGGGGGCGAGCTGTTCTTCCACCTGTCCCGGGAGCGTGTGTTCT
CCGAGGACCGGGCCCGCTTCTATGGCGCTGAGATTGTGTCAGCCCTGGACTACCTGCACTCGGAGAAGAACGTGG
TGTACCGGGACCTCAAGCTGGAGAACCTCATGCTGGACAAGGACGGGCACATTAAGATCACAGACTTCGGGCTGT
GCAAGGAGGGGATCAAGGACGGTGCCACCATGAAGACCTTTTGCGGCACACCTGAGTACCTGGCCCCCGAGGTGC
TGGAGGACAATGACTACGGCCGTGCAGTGGACTGGTGGGGGCTGGGCGTGGTCATGTACGAGATGATGTGCGGTC
GCCTGCCCTTCTACAACCAGGACCATGAGAAGCTTTTTGAGCTCATCCTCATGGAGGAGATCCGCTTCCCGCGCA
CGCTTGGTCCCGAGGCCAAGTCCTTGCTTTCAGGGCTGCTCAAGAAGGACCCCAAGCAGAGGCTTGGCGGGGGCT
CCGAGGACGCCAAGGAGATCATGCAGCATCGCTTCTTTGCCGGTATCGTGTGGCAGCACGTGTACGAGAAGAAGC
TCAGCCCACCCTTCAAGCCCCAGGTCACGTCGGAGACTGACACCAGGTATTTTGATGAGGAGTTCACGGCCCAGA
TGATCACCATCACACCACCTGACCAAGATGACAGCATGGAGTGTGTGGACAGCGAGCGCAGGCCCCACTTCCCCC
AGTTCTCCTACTCGGCCAGCGGCACGGCCTAG'''.replace('\n',''))
gene_capped = first_overhang + sapI_seqplusone + akt + sapI_seqplusone.reverse_complement() + last_overhang
optimize_gene(gene_capped,
             block_size_range = [165,175])



Some regions are medium fidelity.


([[0, 174, 1463],
  [0, 161, 342, 1463],
  [0, 321, 505, 1463],
  [0, 491, 670, 1463],
  [0, 657, 829, 1463],
  [0, 813, 1000, 1463],
  [0, 985, 1164, 1463],
  [0, 1150, 1330, 1463],
  [0, 1296, 1463]],
 [['CGTC', 'TTCT', 'GCAT'],
  ['CGTC', 'TCCC', 'AAGA', 'GCAT'],
  ['CGTC', 'CAGA', 'TCCT', 'GCAT'],
  ['CGTC', 'TTTC', 'ACGA', 'GCAT'],
  ['CGTC', 'TCTT', 'GGGA', 'GCAT'],
  ['CGTC', 'AAGA', 'TGGA', 'GCAT'],
  ['CGTC', 'ACTA', 'AAGA', 'GCAT'],
  ['CGTC', 'TTTC', 'AGGA', 'GCAT'],
  ['CGTC', 'GTCA', 'GCAT']],
 [0.9693578138231524,
  0.9473011537441497,
  0.9493176714669691,
  0.9494231834409504,
  0.9473011537441497,
  0.9490519875816659,
  0.9488266764221456,
  0.9502978746717136,
  0.9672689252729983],
 [174, 185, 188, 183, 176, 191, 183, 184, 167],
 [[0, 1], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2]])

In [195]:
# CFTR portion - c tag SUPERBLOCK1
cftr = Seq('''ATGCAGAGGTCGCCTCTGGAAAAGGCCAGCGTTGTCTCCAAACTTTTTTTCAGCTGGACCAGACCAATTTTGAGGA
AAGGATACAGACAGCGCCTGGAATTGTCAGACATATACCAAATCCCTTCTGTTGATTCTGCTGACAATCTATCTGAAAAATTGGAAAGAG
AATGGGATAGAGAGCTGGCTTCAAAGAAAAATCCTAAACTCATTAATGCCCTTCGGCGATGTTTTTTCTGGAGATTTATGTTCTATGGAA
TCTTTTTATATTTAGGGGAAGTCACCAAAGCAGTACAGCCTCTCTTACTGGGAAGAATCATAGCTTCCTATGACCCGGATAACAAGGAGG
AACGCTCTATCGCGATTTATCTAGGCATAGGCTTATGCCTTCTCTTTATTGTGAGGACACTGCTCCTACACCCAGCCATTTTTGGCCTTC
ATCACATTGGAATGCAGATGAGAATAGCTATGTTTAGTTTGATTTATAAGAAGACTTTAAAGCTGTCAAGCCGTGTTCTAGATAAAATAA
GTATTGGACAACTTGTTAGTCTCCTTTCCAACAACCTGAACAAATTTGATGAAGGACTTGCATTGGCACATTTCGTGTGGATCGCTCCTT
TGCAAGTGGCACTCCTCATGGGGCTAATCTGGGAGTTGTTACAGGCGTCTGCCTTCTGTGGACTTGGTTTCCTGATAGTCCTTGCCCTTT
TTCAGGCTGGGCTAGGGAGAATGATGATGAAGTACAGAGATCAGAGAGCTGGGAAGATCAGTGAAAGACTTGTGATTACCTCAGAAATGA
TTGAAAATATCCAATCTGTTAAGGCATACTGCTGGGAAGAAGCAATGGAAAAAATGATTGAAAACTTAAGACAAACAGAACTGAAACTGA
CTCGGAAGGCAGCCTATGTGAGATACTTCAATAGCTCAGCCTTCTTCTTCTCAGGGTTCTTTGTGGTGTTTTTATCTGTGCTTCCCTATG
CACTAATCAAAGGAATCATCCTCCGGAAAATATTCACCACCATCTCATTCTGCATTGTTCTGCGCATGGCGGTCACTCGGCAATTTCCCT
GGGCTGTACAAACATGGTATGACTCTCTTGGAGCAATAAACAAAATACAGGATTTCTTACAAAAGCAAGAATATAAGACATTGGAATATA
ACTTAACGACTACAGAAGTAGTGATGGAGAATGTAACAGCCTTCTGGGAGGAGGGATTTGGGGAATTATTTGAGAAAGCAAAACAAAACA
ATAACAATAGAAAAACTTCTAATGGTGATGACAGCCTCTTCTTCAGTAATTTCTCACTTCTTGGTACTCCTGTCCTGAAAGATATTAATT
TCAAGATAGAAAGAGGACAGTTGTTGGCGGTTGCTGGATCCACTGGAGCAGGCAAGACTTCACTTCTAATGGTGATTATGGGAGAACTGG
AGCCTTCAGAGGGTAAAATTAAGCACAGTGGAAGAATTTCATTCTGTTCTCAGTTTTCCTGGATTATGCCTGGCACCATTAAAGAAAATA
TCATCTTTGGTGTTTCCTAT'''.replace('\n',''))
gene_capped = first_overhang + sapI_seqplusone + cftr + sapI_seqplusone.reverse_complement() + last_overhang
optimize_gene(gene_capped,
             block_size_range = [165,175])



Some regions are medium fidelity.


([[0, 176, 1556],
  [0, 170, 352, 1556],
  [0, 343, 523, 1556],
  [0, 510, 697, 1556],
  [0, 681, 875, 1556],
  [0, 855, 1045, 1556],
  [0, 1030, 1216, 1556],
  [0, 1209, 1386, 1556],
  [0, 1382, 1556]],
 [['CGTC', 'AGAA', 'GCAT'],
  ['CGTC', 'GGAA', 'AGGA', 'GCAT'],
  ['CGTC', 'CGGA', 'TTCT', 'GCAT'],
  ['CGTC', 'CTGT', 'TCCT', 'GCAT'],
  ['CGTC', 'TTCT', 'AAGA', 'GCAT'],
  ['CGTC', 'GAAA', 'TTCT', 'GCAT'],
  ['CGTC', 'TCTC', 'AGGA', 'GCAT'],
  ['CGTC', 'TTCT', 'TCCA', 'GCAT'],
  ['CGTC', 'TGGA', 'GCAT']],
 [0.9693578138231524,
  0.9503362656703902,
  0.9481735534989284,
  0.9484119248465395,
  0.9504530934571519,
  0.949841214886651,
  0.9493955988181414,
  0.9488457615956638,
  0.9683339822030734],
 [176, 186, 184, 191, 198, 194, 190, 181, 174],
 [[0, 1], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2]])

In [245]:
pten=Seq('atgacagccatcatcaaagagatcgttagcagaaacaaaaggagatatcaagaggatggattcgacttagacttgacctatatttatccaaacattattgctatgggatttcctgcagaaagacttgaaggcgtatacaggaacaatattgatgatgtagtaaggtttttggattcaaagcataaaaaccattacaagatatacaatctttgtgctgaaagacattatgacaccgccaaatttaattgcagagttgcacaatatccttttgaagaccataacccaccacagctagaacttatcaaacccttttgtgaagatcttgaccaatggctaagtgaagatgacaatcatgttgcagcaattcactgtaaagctggaaagggacgaactggtgtaatgatatgtgcatatttattacatcggggcaaatttttaaaggcacaagaggccctagatttctatggggaagtaaggaccagagacaaaaagggagtaactattcccagtcagaggcgctatgtgtattattatagctacctgttaaagaatcatctggattatagaccagtggcactgttgtttcacaagatgatgtttgaaactattccaatgttcagtggcggaacttgcaatcctcagtttgtggtctgccagctaaaggtgaagatatattcctccaattcaggacccacacgacgggaagacaagttcatgtactttgagttccctcagccgttacctgtgtgtggtgatatcaaagtagagttcttccacaaacagaacaagatgctaaaaaaggacaaaatgtttcacttttgggtaaatacattcttcataccaggaccagaggaaacctcagaaaaagtagaaaatggaagtctatgtgatcaagaaatcgatagcatttgcagtatagagcgtgcagataatgacaaggaatatctagtacttactttaacaaaaaatgatcttgacaaagcaaataaagacaaagccaaccgatacttttctccaaattttaaggtgaagctgtacttcacaaaaacagtagaggagccgtcaaatccagaggctagcagttcaacttctgtaacaccagatgttagtgacaatgaacctgatcattatagatattctgacaccactgactctgatccagagaatgaaccttttgatgaagatcagcatacacaaattacaaaagtctag'.upper())
gene_capped = first_overhang + sapI_seqplusone + pten + sapI_seqplusone.reverse_complement() + last_overhang
optimize_gene(gene_capped,
             block_size_range = [165,175])



Some regions are medium fidelity.


([[0, 181, 1232],
  [0, 169, 352, 1232],
  [0, 340, 530, 1232],
  [0, 514, 698, 1232],
  [0, 687, 873, 1232],
  [0, 862, 1048, 1232],
  [0, 1035, 1224, 1232],
  [0, 1055, 1232]],
 [['CGTC', 'TGGA', 'GCAT'],
  ['CGTC', 'TAGT', 'AAGA', 'GCAT'],
  ['CGTC', 'AATG', 'CTAT', 'GCAT'],
  ['CGTC', 'TTCC', 'AGGA', 'GCAT'],
  ['CGTC', 'TCCT', 'GAAA', 'GCAT'],
  ['CGTC', 'AGGA', 'ACTT', 'GCAT'],
  ['CGTC', 'AAGG', 'GGAA', 'GCAT'],
  ['CGTC', 'AAAA', 'GCAT']],
 [0.9683339822030734,
  0.9488266764221456,
  0.9458903938126513,
  0.9503362656703902,
  0.9502978746717136,
  0.9496812404894291,
  0.948351852390686,
  0.9682170579431125],
 [181, 187, 194, 188, 190, 190, 193, 177],
 [[0, 1], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2]])

In [255]:
mek=Seq('atgcccaagaagaagccgacgcccatccagctgaacccggcccccgacggctctgcagttaacgggaccagctctgcgGAAACAaacttggaggccttgcagaagaagctggaggagctagagcttgatgagcagcagcgaaagcgccttgaggcctttcttacccagaagcagaaggtgggagaactgaaggatgacgactttgagaagatcagtgagctgggggctggcaatggcggtgtggtgttcaaggtGtcccacaagccttctggcctggtcatggccagaaagctaattcatctggagatcaaacccgcaatccggaaccagatcataagggagctgcaggttctgcatgagtgcaactctccgtacatcgtgggcttctatggtgcgttctacagcgatggcgagatcagtatctgcatggagcacatggatggaggttctctggatcaagtcctgaagaaagctggaagaattcctgaacaaattttaggaaaagttagcattgctgtaataaaaggcctgacatatctgagggagaagcacaagatcatgcacagagatgtcaagccctccaacatcctagtcaactcccgtggggagatcaagctctgtgactttggggtcagcgggcagctcatcgactccatggccaactccttcgtgggcacaaggtcctacatgtcgccagaaagactccaggggactcattactctgtgcagtcagacatctggagcatgggactgtctctggtagagatggcggttgggaggtatcccatccctcctccagatgccaaggagctggagctgatgtttgggtgccaggtggaaggagatgcggctgagacAccacccaggccaaggacccccgggaggccccttagctcatacggaatggacagccgacctcccatggcaatttttgagttgttggattacatagtcaacgagcctcctccaaaactgcccagtggagtgttcagtctggaatttcaagattttgtgaataaatgcttaataaaaaaccccgcagagagagcagatttgaagcaactcatggttcatgcttttatcaagagatctgatgctgaggaagtggattttgcaggttggctctgctccaccatcggccttaaccagcccagcacaccaacccatgctgctggcgtcTAG'.upper())
gene_capped = first_overhang + sapI_seqplusone + mek + sapI_seqplusone.reverse_complement() + last_overhang
optimize_gene(gene_capped,
             block_size_range = [165,175])



Some regions are medium fidelity.


([[0, 178, 1202],
  [0, 169, 347, 1202],
  [0, 334, 521, 1202],
  [0, 510, 693, 1202],
  [0, 679, 862, 1202],
  [0, 851, 1040, 1202],
  [0, 1029, 1202]],
 [['CGTC', 'AGAA', 'GCAT'],
  ['CGTC', 'TTCT', 'AAGG', 'GCAT'],
  ['CGTC', 'GGAA', 'CATT', 'GCAT'],
  ['CGTC', 'GGAA', 'TCCT', 'GCAT'],
  ['CGTC', 'TCGT', 'CTGA', 'GCAT'],
  ['CGTC', 'AGGA', 'AAAA', 'GCAT'],
  ['CGTC', 'AAAT', 'GCAT']],
 [0.9693578138231524,
  0.9487182377454961,
  0.9481545990377396,
  0.9503362656703902,
  0.9493845081483289,
  0.9491419652869855,
  0.968299283846298],
 [178, 182, 191, 187, 187, 193, 173],
 [[0, 1], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2], [1, 2]])

In [256]:
oligo_array,amp_primers,gblocks = write_oligo_library({'HIST1H1E':hist1h1e,
                                                       'RPS19':rps19,
                                                       'SPOP':spop,
                                                       'AKT':akt,
                                                       'CFTR':cftr,
                                                       'PTEN':pten,
                                                       'MEK':mek},
                                                      oligo_file='./bsaI_test/evan_oligos.csv',
                                                      primer_file='./bsaI_test/evan_primers.tsv',
                                                      gbl_file='./bsaI_test/evan_gblocks.tsv',
                                                      all_blocks=False,
                                                      blocks_to_include=[[2],[2],[3],[5],[9],[3],[3]],
                                                      block_size_range=[165,175])


Processing gene 1
Gene has no SapI or BsaI sites! Performing GoldenGate optimization...
All regions are high fidelity!
{'Optimum Breakpoints': [[0, 183, 680],
                         [0, 166, 353, 680],
                         [0, 336, 521, 680],
                         [0, 506, 680]],
 'Optimum Overlaps': [['CGTC', 'TCTT', 'GCAT'],
                      ['CGTC', 'AGGA', 'GGAA', 'GCAT'],
                      ['CGTC', 'AAGA', 'GAAA', 'GCAT'],
                      ['CGTC', 'TGGA', 'GCAT']],
 'Optimum Scores': [0.9695086923749419,
                    0.9503362656703902,
                    0.9500476572285086,
                    0.9683339822030734]}
[0, 166]
[353, 680]
Processing gene 2
Gene has no SapI or BsaI sites! Performing GoldenGate optimization...
Some regions are medium fidelity.
{'Optimum Breakpoints': [[0, 168, 458], [0, 160, 341, 458], [0, 286, 458]],
 'Optimum Overlaps': [['CGTC', 'TTCT', 'GCAT'],
                      ['CGTC', 'AGAA', 'GAAA', 'GCAT'],
                  

In [227]:
braf=Seq('TCTAGCGAGGACCGCAACAGAATGAAGACTCTCGGTAGAAGAgactcgagtGACGACTGGGAAATACCAGACGGTCAAATCACGGTCGGTCAGCGGATCGGATCAGGCTCCTTCGGGACTGTATATAAAGGTAAATGGCACGGCGACGTTGCGGTCAAAATGCTGAACGTTACAGCACCAACCCCCCAACAGCTTCAAGCGTTCAAGAACGAAGTCGGGGTACTGCGCAAAACTCGGCATGTCAATATATTGCTGTTCATGGGTTACTCAACCAAGCCTCAACTCGCCATAGTTACCCAGTGGTGTGAAGGCAGCTCACTTTACCATCACCTGCACATAATAGAAACAAAGTTCGAGATGATCAAACTCATTGACATTGCGCGGCAAACAGCGCAGGGGATGGACTATCTGCATGCAAAGTCTATTATCCATAGAGATCTCAAGTCAAACAACATATTCTTGCACGAGGATCTCACCGTTAAAATTGGAGACTTTGGCCTGGCAACTGTTAAGTCACGGTGGAGTGGGTCACATCAGTTCGAGCAGCTGTCCGGCAGCATACTGTGGATGGCTCCAGAGGTCATTCGCATGCAGGATAAGAACCCTTATTCTTTCCAATCCGATGTTTATGCATTTGGCATCGTCCTCTATGAGCTGATGACCGGACAACTCCCTTACAGCAACATCAACAATCGAGATCAGATCATCTTCATGGTCGGGCGAGGATACCTCAGCCCCGATCTCTCAAAGGTTCGATCAAATTGCCCTAAAGCGATGAAACGGCTTATGGCGGAGTGTTTGAAAAAGAAACGCGACGAACGCCCTTTGTTCCCTCAAATCTTGGCATCAATCGAGTTGCTGGCTAGAAGTCTTCCAAAAATACACAGAAGCGCATCCGAGCCAAGCCTCAATCGGGCAGGATTCCAGACCGAGGATTTCTCTCTTTATGCCTGCGCATCTCCAAAGACACCCATACAGGCCGGCGGGTACggCgcCtttcctgtgcacTAG'.upper())
mek=Seq('atgcccaagaagaagccgacgcccatccagctgaacccggcccccgacggctctgcagttaacgggaccagctctgcgGAAACAaacttggaggccttgcagaagaagctggaggagctagagcttgatgagcagcagcgaaagcgccttgaggcctttcttacccagaagcagaaggtgggagaactgaaggatgacgactttgagaagatcagtgagctgggggctggcaatggcggtgtggtgttcaaggtGtcccacaagccttctggcctggtcatggccagaaagctaattcatctggagatcaaacccgcaatccggaaccagatcataagggagctgcaggttctgcatgagtgcaactctccgtacatcgtgggcttctatggtgcgttctacagcgatggcgagatcagtatctgcatggagcacatggatggaggttctctggatcaagtcctgaagaaagctggaagaattcctgaacaaattttaggaaaagttagcattgctgtaataaaaggcctgacatatctgagggagaagcacaagatcatgcacagagatgtcaagccctccaacatcctagtcaactcccgtggggagatcaagctctgtgactttggggtcagcgggcagctcatcgactccatggccaactccttcgtgggcacaaggtcctacatgtcgccagaaagactccaggggactcattactctgtgcagtcagacatctggagcatgggactgtctctggtagagatggcggttgggaggtatcccatccctcctccagatgccaaggagctggagctgatgtttgggtgccaggtggaaggagatgcggctgagacAccacccaggccaaggacccccgggaggccccttagctcatacggaatggacagccgacctcccatggcaatttttgagttgttggattacatagtcaacgagcctcctccaaaactgcccagtggagtgttcagtctggaatttcaagattttgtgaataaatgcttaataaaaaaccccgcagagagagcagatttgaagcaactcatggttcatgcttttatcaagagatctgatgctgaggaagtggattttgcaggttggctctgctccaccatcggccttaaccagcccagcacaccaacccatgctgctggcgtcTAG'.upper())
egfr=Seq('AGGCGCCACATCGTTCGGAAGCGCACGCTGCGGAGGCTGCTGCAGGAGAGGGAGCTTGTGGAGCCTCTTACACCCAGTGGAGAAGCTCCCAACCAAGCTCTCTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGTGCTGGGCTCCGGTGCGTTCGGCACGGTGTATAAGGGACTCTGGATCCCAGAAGGTGAGAAAGTTAAAATTCCCGTCGCTATCAAGGAATTAAGAGAAGCAACATCTCCGAAAGCCAACAAGGAAATCCTCGATGAAGCCTACGTGATGGCCAGCGTGGACAACCCCCACGTGTGCCGCCTGCTGGGCATCTGCCTCACCTCCACCGTGCAACTCATCACGCAGCTCATGCCCTTCGGCTGCCTCCTGGACTATGTCCGGGAACACAAAGACAATATTGGCTCCCAGTACCTGCTCAACTGGTGTGTGCAGATCGCAAAGGGCATGAACTACTTGGAGGACCGTCGCTTGGTGCACCGCGACCTGGCAGCCAGGAACGTACTGGTGAAAACACCGCAGCATGTCAAGATCACAGATTTTGGGCTGGCCAAACTGCTGGGTGCGGAAGAGAAAGAATACCATGCAGAAGGAGGCAAAGTGCCTATCAAGTGGATGGCATTGGAATCAATTTTACACAGAATCTATACCCACCAGAGTGATGTCTGGAGCTACGGGGTGACCGTTTGGGAGTTGATGACCTTTGGATCCAAGCCATATGACGGAATCCCTGCCAGCGAGATCTCCTCCATCCTGGAGAAAGGAGAACGCCTCCCTCAGCCACCCATATGTACCATCGATGTCTACATGATCATGGTCAAGTGCTGGATGATAGACGCAGATAGTCGCCCAAAGTTCCGTGAGTTGATCATCGAATTCTCCAAAATGGCCCGAGATCCCCAGCGCTACCTTGTCATTCAGGGGGATGAAAGAATGCATTTGCCAAGTCCTACAGACTCCAACTTCTACCGTGCCCTGATGGATGAAGAAGACATGGACGACGTGGTGGATGCCGACGAGTACCTCATCCCACAGCAGGGCTTCTTCAGCAGCCCCTCCACGTCACGGACTCCCCTCCTGAGCTCTCTGAGTGCAACCAGCAACAATTCCACCGTGGCTTGCATTGATAGAAATGGGCTGCAAAGCTGTCCCATCAAGGAAGACAGCTTCTTGCAGCGATACAGCTCAGACCCCACAGGCGCCTTGACTGAGGACAGCATAGACGACACCTTCCTCCCAGTGCCTGAATACATAAACCAGTCCGTTCCCAAAAGGCCCGCTGGCTCTGTGCAGAATCCTGTCTATCACAATCAGCCTCTGAACCCCGCGCCCAGCAGAGATCCACACTACCAGGACCCCCACAGCACTGCAGTGGGCAACCCCGAGTATCTCAACACTGTCCAGCCCACCTGTGTCAACAGCACATTCGACAGCCCTGCCCACTGGGCCCAGAAAGGCAGCCACCAAATTAGCCTGGACAACCCTGACTACCAGCAGGACTTCTTTCCCAAGGAAGCCAAGCCAAATGGCATCTTTAAGGGCTCCACAGCTGAAAATGCAGAATACCTAAGGGTCGCGCCACAAAGCAGTGAATTTATTGGAGCAACG'.upper())

oligo_array,amp_primers,gblocks = write_oligo_library({'PTEN':pten,
                                                       'BRAF':braf,
                                                       'MEK':mek,
                                                       'EGFR':egfr},
                                                      oligo_file='./bsaI_test/circRNApilot_oligos.csv',
                                                      primer_file='./bsaI_test/circRNApilot_primers.tsv',
                                                      gbl_file='./bsaI_test/circRNApilot_gblocks.tsv',
                                                      block_size_range=[150,170])


Processing gene 1
Gene has no SapI or BsaI sites! Performing GoldenGate optimization...
Some regions are medium fidelity.
{'Optimum Breakpoints': [[0, 158, 1232],
                         [0, 150, 319, 1232],
                         [0, 305, 470, 1232],
                         [0, 457, 628, 1232],
                         [0, 612, 781, 1232],
                         [0, 772, 938, 1232],
                         [0, 918, 1091, 1232],
                         [0, 1079, 1232]],
 'Optimum Overlaps': [['CGTC', 'TATT', 'GCAT'],
                      ['CGTC', 'AGGA', 'CCTT', 'GCAT'],
                      ['CGTC', 'AGAA', 'TTTC', 'GCAT'],
                      ['CGTC', 'AAGA', 'TCAG', 'GCAT'],
                      ['CGTC', 'GAAA', 'TCTT', 'GCAT'],
                      ['CGTC', 'AAGT', 'AGAT', 'GCAT'],
                      ['CGTC', 'ATTT', 'CAGT', 'GCAT'],
                      ['CGTC', 'TCCA', 'GCAT']],
 'Optimum Scores': [0.9671470761342998,
                    0.9491743576313715,
    

In [274]:
mek=Seq('ATGCCCAAGAAAAAACCTACACCCATACAACTGAATCCCGCACCAGACGGATCAGCAGTTAATGGAACGTCCTCTGCCGAAACAAACCTTGAGGCATTGCAGAAAAAACTTGAAGAATTGGAGCTTGATGAACAGCAGCGGAAAAGATTGGAGGCTTTCCTTACCCAGAAGCAGAAGGTTGGAGAGCTTAAAGACGATGACTTTGAAAAAATCTCTGAGCTTGGAGCCGGAAATGGCGGCGTTGTGTTCAAAGTTAGCCACAAACCTTCAGGGCTTGTTATGGCCAGAAAACTTATTCATCTGGAAATCAAGCCAGCAATCcggaaccagatcataagggagctgcaggttctgcatgagtgcaactctccgtacatcgtgggcttctatggtgcgttctacagcgatggcgagatcagtatctgcatggagcacatggatggaggttctctggatcaagtcctgaagaaagctggaagaattcctgaacaaattttaggaaaagttagcattGCGGTAATAAAGGGATTGACGTATCTGAGAGAAAAGCACAAAATTATGCATAGAGATGTCAAGCCATCCAATATTCTCGTTAATTCCAGGGGTGAGATTAAGCTGTGCGATTTTGGTGTGTCCGGTCAACTCATCGATTCCATGGCAAATTCATTCGTTGGTACTAGGTCTTACATGAGCCCCGAGAGGCTTCAGGGCACTCACTATAGCGTTCAGAGCGATATCTGGAGCATGGGGTTGAGTCTGGTCGAGATGGCAGTAGGGAGGTATCCAATACCACCGCCCGACGCTAAAGAGCTTGAGCTCATGTTCGGATGCCAGGTTGAAGGCGACGCGGCTGAGACACCGCCACGACCCCGCACACCGGGGCGACCATTGAGTAGCTATGGTATGGACTCTCGCCCTCCAATGGCGATCTTCGAGCTGCTGGACTATATAGTTAACGAACCCCCACCTAAATTGCCTTCCGGGGTTTTTTCTCTCGAATTTCAAGACTTCGTCAATAAATGCCTCATAAAAAATCCCGCCGAACGCGCTGATCTGAAGCAGCTTATGGTTCATGCGTTCATAAAGCGAAGCGATGCCGAGGAGGTTGATTTCGCAGGTTGGTTGTGTTCAACTATAGGACTCAACCAACCCAGCACACCGACTCACGCCGCAGGGGTCTAG'.upper())

oligo_array,amp_primers,gblocks = write_oligo_library({'MEK':mek},
                                                      oligo_file='./bsaI_test/test_oligos_110223.csv',
                                                      primer_file='./bsaI_test/test_primer_110223.tsv',
                                                      gbl_file='./bsaI_test/test_gbl_110223.tsv',
                                                      all_blocks=False,
                                                      blocks_to_include=[[3]],
                                                      block_size_range=[165,175])


Processing gene 1
Gene has no SapI or BsaI sites! Performing GoldenGate optimization...
Some regions are medium fidelity.
{'Optimum Breakpoints': [[0, 178, 1202],
                         [0, 169, 347, 1202],
                         [0, 334, 521, 1202],
                         [0, 509, 693, 1202],
                         [0, 679, 862, 1202],
                         [0, 850, 1043, 1202],
                         [0, 1029, 1202]],
 'Optimum Overlaps': [['CGTC', 'AGAA', 'GCAT'],
                      ['CGTC', 'TCCT', 'AAGG', 'GCAT'],
                      ['CGTC', 'GGAA', 'CATT', 'GCAT'],
                      ['CGTC', 'AGGA', 'TCTT', 'GCAT'],
                      ['CGTC', 'TCGT', 'CTGA', 'GCAT'],
                      ['CGTC', 'AAGG', 'AAAT', 'GCAT'],
                      ['CGTC', 'AAAT', 'GCAT']],
 'Optimum Scores': [0.9693578138231524,
                    0.9491743576313715,
                    0.9481545990377396,
                    0.9509100474180536,
                    0.9493